# Variable Selection Study - Linear Regression
Comprehensive application of multiple variable selection methods.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.data_prep import prepare_data
from src.features import run_feature_engineering

## Load Data

In [ ]:
df = prepare_data()
df = run_feature_engineering(df)

## Define Features

In [ ]:
features = [
    'price','freight_value','delivery_time','lag_price','lag_sales',
    'discount_ratio','month','day_of_week'
]

X = df[features].fillna(0)
y = df['profit']

## OLS (Baseline)

In [ ]:
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

## Backward Elimination

In [ ]:
def backward_elimination(X, y, threshold=0.05):
    X = sm.add_constant(X)
    while True:
        model = sm.OLS(y, X).fit()
        pvalues = model.pvalues.drop('const')
        max_p = pvalues.max()

        if max_p > threshold:
            remove_var = pvalues.idxmax()
            X = X.drop(columns=[remove_var])
        else:
            break

    return model, X

model_be, X_be = backward_elimination(X, y)
print(model_be.summary())

## VIF Analysis

In [ ]:
vif = pd.DataFrame()
vif['feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif.sort_values('VIF', ascending=False)

## Lasso

In [ ]:
lasso = LassoCV(cv=5)
lasso.fit(X, y)
pd.Series(lasso.coef_, index=X.columns).sort_values()

## Ridge

In [ ]:
ridge = RidgeCV(cv=5)
ridge.fit(X, y)
pd.Series(ridge.coef_, index=X.columns).sort_values()

## Random Forest Importance

In [ ]:
rf = RandomForestRegressor()
rf.fit(X, y)
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

## Final Comparison Table

In [ ]:
results = pd.DataFrame({
    'feature': X.columns,
    'lasso': lasso.coef_,
    'ridge': ridge.coef_,
    'rf_importance': rf.feature_importances_
})

results.sort_values('rf_importance', ascending=False)